# ASR + SOAP + Speaker Diarization (LLM-based)

- Transcribe audio with faster-whisper (CPU)
- Generate SOAP notes using HF Llama
- Generate speaker diarization from transcript using the same LLM

In [3]:

import argparse
import gc
import os
from pathlib import Path

from dotenv import load_dotenv

from pydub import AudioSegment
from faster_whisper import WhisperModel
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import torch

# Audio-based diarization
import whisperx
from whisperx.diarize import DiarizationPipeline

# Load environment variables from .env (if present)
load_dotenv()


def cleanup_all():
    'Run GC to keep memory usage steady on small machines.'
    gc.collect()


def preprocess_audio(audio_path: Path, out_dir: Path) -> Path:
    'Convert audio to mono 16kHz WAV for ASR. Returns processed WAV path.'
    audio = AudioSegment.from_file(audio_path)
    audio = audio.set_channels(1).set_frame_rate(16000)

    processed_path = out_dir / f"{audio_path.stem}_mono16k.wav"
    audio.export(processed_path, format="wav")
    return processed_path


def transcribe_audio(wav_path: Path, out_dir: Path, model_size: str = "small") -> Path:
    'Transcribe WAV with faster-whisper (CPU). Returns transcript file path.'
    asr_model = WhisperModel(model_size, device="cpu", compute_type="int8")
    segments, _info = asr_model.transcribe(str(wav_path), beam_size=5)

    transcript_text = " ".join(seg.text.strip() for seg in segments)

    transcript_path = out_dir / "full_transcript.txt"
    transcript_path.write_text(transcript_text.strip(), encoding="utf-8")

    del asr_model
    cleanup_all()

    return transcript_path


def _build_chat_model(repo_id: str, max_new_tokens: int = 1024):
    'Create HF chat model wrapper. Separating this removes Pylance warnings.'
    token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if not token:
        raise RuntimeError("HUGGINGFACEHUB_API_TOKEN is not set.")

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        task="text-generation",
        max_new_tokens=max_new_tokens,
        temperature=0.01,
    )

    return ChatHuggingFace(llm=llm)


def summarize_soap(transcript_text: str, repo_id: str, max_new_tokens: int = 1024) -> str:
    'Generate SOAP notes using a HF chat model.'
    chat_model = _build_chat_model(repo_id, max_new_tokens=max_new_tokens)

    soap_template = [
        (
            "system",
            "You are a medical scribe. Convert the transcript into a formal SOAP note. "
            "If a section has no information, write 'Not mentioned'. Output exactly:\n"
            "S: ...\nO: ...\nA: ...\nP: ...",
        ),
        ("human", "Transcript:\n{transcript}"),
    ]

    prompt = ChatPromptTemplate.from_messages(soap_template)
    chain = prompt | chat_model | StrOutputParser()

    return chain.invoke({"transcript": transcript_text})


def diarize_audio(audio_path: Path, out_dir: Path, model_size: str = "small") -> Path:
    '''
    True audio-based diarization using WhisperX + Pyannote.
    Writes speaker-labeled transcript to speaker_diarization.txt
    '''
    hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if not hf_token:
        raise RuntimeError("HUGGINGFACEHUB_API_TOKEN is not set (required for diarization).")

    device = "cpu"  # WhisperX diarization is most stable on CPU for macOS
    compute_type = "int8"

    # Load audio
    audio = whisperx.load_audio(str(audio_path))

    # PyTorch 2.6+ defaults to weights_only=True in some paths; allow trusted checkpoint objects.
    os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
    try:
        from omegaconf.listconfig import ListConfig
        from omegaconf.dictconfig import DictConfig
        torch.serialization.add_safe_globals([ListConfig, DictConfig])
    except Exception:
        pass

    # 1) Transcribe with WhisperX (needed for word-level timestamps)
    # Use Silero VAD here to avoid pyannote VAD checkpoint-loading issues.
    model = whisperx.load_model(model_size, device, compute_type=compute_type, vad_method="silero")
    result = model.transcribe(audio)

    # 2) Align for accurate word timestamps
    align_model, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    result = whisperx.align(result["segments"], align_model, metadata, audio, device)

    # 3) Diarize with Pyannote
    diarize_model = DiarizationPipeline(use_auth_token=hf_token, device=device)
    diarize_segments = diarize_model(audio)

    # 4) Assign speakers to words/segments
    result = whisperx.assign_word_speakers(diarize_segments, result)

    # 5) Write speaker-labeled transcript
    lines = []
    for seg in result["segments"]:
        speaker = seg.get("speaker", "SPEAKER_0")
        text = seg.get("text", "").strip()
        if text:
            lines.append(f"{speaker}: {text}")

    diar_path = out_dir / "speaker_diarization.txt"
    diar_path.write_text("\n".join(lines), encoding="utf-8")
    return diar_path


In [4]:

# ---- Run pipeline ----

# Update these paths for your file
audio_path = "./audio/doc_patient_convo.mp3"
out_dir = "./output/v2"

hf_model = "meta-llama/Llama-3.1-8B-Instruct"
asr_model = "small"

from pathlib import Path

def run_pipeline(audio_path: str, out_dir: str, hf_model: str, asr_model: str = "small"):
    audio_path = Path(audio_path).expanduser().resolve()
    out_dir = Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    processed_wav = preprocess_audio(audio_path, out_dir)
    transcript_path = transcribe_audio(processed_wav, out_dir, model_size=asr_model)

    transcript_text = transcript_path.read_text(encoding="utf-8")
    soap_notes = summarize_soap(transcript_text, repo_id=hf_model)

    soap_path = out_dir / "soap_notes.txt"
    soap_path.write_text(soap_notes, encoding="utf-8")

    # Audio-based diarization (true speaker diarization)
    speaker_path = diarize_audio(processed_wav, out_dir, model_size=asr_model)

    print(f"Transcript saved: {transcript_path}")
    print(f"SOAP notes saved: {soap_path}")
    print(f"Speaker diarization saved: {speaker_path}")


run_pipeline(audio_path, out_dir, hf_model, asr_model)


/home/tckleme-dev/.local/share/mise/installs/python/3.10.19/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


2026-02-15 16:32:22 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-02-15 16:32:22 - whisperx.vads.silero - INFO - Performing voice activity detection using Silero...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/tckleme-dev/.cache/torch/hub/master.zip
2026-02-15 16:32:29 - whisperx.asr - INFO - Detected language: en (0.99) in first 30s of audio
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /home/tckleme-dev/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 360M/360M [00:09<00:00, 39.4MB/s]


2026-02-15 16:33:28 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-3.1


/home/tckleme-dev/.local/share/mise/installs/python/3.10.19/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
/home/tckleme-dev/.local/share/mise/installs/python/3.10.19/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()
/home/tckleme-dev/.local/share/mise/installs/python/3.10.19/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:1

Transcript saved: /home/tckleme-dev/Documents/College/fyp/ml-service/output/v2/full_transcript.txt
SOAP notes saved: /home/tckleme-dev/Documents/College/fyp/ml-service/output/v2/soap_notes.txt
Speaker diarization saved: /home/tckleme-dev/Documents/College/fyp/ml-service/output/v2/speaker_diarization.txt
